# Step 4-A: DQN 신경망 구조 및 액션 마스킹 설계 가이드

## 1. 네트워크 아키텍처 스펙 (Architecture Specifications)

보드의 공간 기하학(말의 위치, 가로/세로 벽)을 보존하기 위해 3단계 Conv 블록과 2단계 Fully-Connected 블록을 구성합니다.

### A. 합성곱 계층 (Convolutional Layers)
- **입력 형상**: `(Batch_Size, 6, 9, 9)`
- **Conv 1**: `in_channels=6`, `out_channels=64`, `kernel_size=3`, `padding=1`, `stride=1`
  - 출력 크기: `(B, 64, 9, 9)`
- **Conv 2**: `in_channels=64`, `out_channels=128`, `kernel_size=3`, `padding=1`, `stride=1`
  - 출력 크기: `(B, 128, 9, 9)`
- **Conv 3**: `in_channels=128`, `out_channels=64`, `kernel_size=3`, `padding=1`, `stride=1`
  - 출력 크기: `(B, 64, 9, 9)`
- 각 Conv 계층 뒤에는 활성화 함수 `F.relu`를 적용합니다.

> **팁 (왜 padding=1 인가?)**:
> $O = \lfloor (W - K + 2P)/S \rfloor + 1 = (9 - 3 + 2)/1 + 1 = 9$.
> 즉, 3x3 커널에 패딩 1을 주면 보드의 가장자리 정보가 잘려 나가지 않고 9x9 크기가 그대로 유지됩니다.

### B. 평탄화 및 완전연결 계층 (Fully-Connected Layers)
- **평탄화 (Flatten)**: `(B, 64, 9, 9)` ➡️ `(B, 64 * 9 * 9 = 5184)`
- **Linear 1**: `in_features=5184`, `out_features=256` + `F.relu`
- **Linear 2 (Output Layer)**: `in_features=256`, `out_features=140`
> **주의**: 최종 출력층 뒤에는 ReLU를 붙이지 않습니다! Q값은 상황에 따라 음수가 될 수도 있어야 합니다.

---

## 2. 액션 마스킹 (Action Masking) 원리 및 구현

- **변수 `mask`의 정의**: `(Batch_Size, 140)` 크기의 `torch.bool` 텐서입니다.
  - `mask[b, a] == True`: $b$번째 배치의 상태에서 $a$번 행동이 합법적인 수임
  - `mask[b, a] == False`: $b$번째 배치의 상태에서 $a$번 행동이 불법적인 수임 (벽 충돌, 이동 불가 등)

- **마스킹 수식**:
  $$\text{If } \neg \text{mask}[b, a] \implies Q(s, a) = -10^9$$

- **파이토치 구현 방법**:
  PyTorch의 `tensor.masked_fill(mask_condition, value)` 함수를 사용합니다.
  - `mask`가 주어졌을 때(`if mask is not None:`):
  - `~mask`는 `mask == False`인 위치를 뜻합니다.
  - `q_values = q_values.masked_fill(~mask, -1e9)` 형태로 불법 행동 위치를 매우 작은 음수로 채웁니다.
